# twbt：台股回測工具組（Colab 版）

**第一次使用前**，把這 3 個檔案上傳到 Google 雲端硬碟「我的雲端硬碟」最外層（zip 不用解壓縮）：
- `twbt-step3.zip`：程式
- `twbt_data.zip`：已整理好的資料（只有第一次需要）
- `twbt_colab.ipynb`：這個筆記本

從上到下按 ▶ 執行。資料存在雲端硬碟的 `twbt_data` 資料夾，下次打開會接著用。

## 1. 連接雲端硬碟、準備程式與資料

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, os, shutil, sys, zipfile
DRIVE = '/content/drive/MyDrive'
DATA = f'{DRIVE}/twbt_data'          # 資料長期存這裡
CODE = '/content/twbt'               # 程式放在 Colab 本機，執行比較快
os.makedirs(DATA, exist_ok=True)

# 程式：用雲端硬碟裡「最新上傳」的 twbt-step*.zip
zips = glob.glob(f'{DRIVE}/twbt-step*.zip')
if not zips:
    raise FileNotFoundError('雲端硬碟最外層找不到 twbt-step*.zip，請先上傳')
latest = max(zips, key=os.path.getmtime)
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(latest) as z:
    z.extractall('/content')
print('程式版本：', os.path.basename(latest))

# 資料：第一次從 twbt_data.zip 解開到 twbt_data 資料夾
if not os.path.exists(f'{DATA}/twbt.db') and os.path.exists(f'{DRIVE}/twbt_data.zip'):
    with zipfile.ZipFile(f'{DRIVE}/twbt_data.zip') as z:
        z.extractall(DATA)
    print('已從 twbt_data.zip 匯入資料')

# 資料庫複製到本機來跑（SQLite 直接在雲端硬碟上寫入很慢），跑完再存回去
os.makedirs(f'{CODE}/data', exist_ok=True)
if os.path.exists(f'{DATA}/twbt.db'):
    shutil.copy(f'{DATA}/twbt.db', f'{CODE}/data/twbt.db')
    print('已載入資料庫')
else:
    print('⚠ 找不到資料庫：請上傳 twbt_data.zip，或在第 3 格從頭下載（約 2～3 小時）')

os.chdir(CODE)
sys.path.insert(0, CODE)

## 2. 跑測試（不需連網）
最後看到 `OK` 就代表程式正常。

In [ ]:
!python -m unittest discover tests 2>&1 | tail -3

## 3. 更新資料

只會補最近兩個月和新月份，通常一兩分鐘。完成後會重新計算還原價和模擬序列，並存回雲端硬碟。

如果出現「連線錯誤」或「回應不是 JSON」一直重試，代表 Colab 被證交所擋了，可以停掉這格，回 Claude 那邊請它幫忙更新。

In [ ]:
!python scripts/update_core.py --out {DATA}/adjusted --backup-to {DATA}/twbt.db

## 4. 看看資料

In [ ]:
import pandas as pd
A = f'{DATA}/adjusted'
print(open(f'{A}/long/calibration.txt', encoding='utf-8').read())

for f in ['0050', '00631L', '00675L']:
    x = pd.read_csv(f'{A}/long/{f}_long.csv', index_col=0, parse_dates=True)
    real = x[x.is_synthetic == 0]
    print(f'{f}：{x.index[0].date()} ~ {x.index[-1].date()}，真實資料從 {real.index[0].date()} 起')

m = pd.read_csv(f'{A}/long/TAIEX_long.csv', index_col=0, parse_dates=True)
ax = m['tr_index'].plot(logy=True, figsize=(10, 4), title='TAIEX total return index (log scale, pre-2003 estimated)')

# 回測：買進持有

**共同設定**（每次改完都要重新執行這格）：
- `CAPITAL`：投入金額
- `DISCOUNT`：手續費折扣，1 = 不打折，0.28 = 2.8 折
- `USE_SYNTHETIC`：是否使用上市前的模擬資料（關掉 = 只用真實資料）
- `TW50_EXCESS`：模擬段的 0050、00631L 是否套用「台灣50 贏大盤」的校正差距。關掉 = 保守版，假設上市前台灣50 和大盤表現相同

In [ ]:
#@title 5. 回測共同設定
CAPITAL = 1000000 #@param {type:"integer"}
DISCOUNT = 1.0 #@param {type:"number"}
USE_SYNTHETIC = True #@param {type:"boolean"}
TW50_EXCESS = True #@param {type:"boolean"}

import warnings; warnings.simplefilter('ignore')
import matplotlib.pyplot as plt
import pandas as pd
from twbt.backtest import BuyHold, TaiwanCost, describe, worst_best, win_rate
from twbt.backtest.report import fmt_stats, fmt_rows
from twbt.backtest.plots import plot_equity, plot_windows

bh = BuyHold(f'{DATA}/adjusted', use_synthetic=USE_SYNTHETIC, capital=CAPITAL,
             cost=TaiwanCost(discount=DISCOUNT), tw50_excess=TW50_EXCESS)
print(f'資料期間：{bh.rets.index[0].date()} ~ {bh.rets.index[-1].date()}')

In [ ]:
#@title 6. 自訂區間
START = '2000-01-01' #@param {type:"date"}
END = '2010-12-31' #@param {type:"date"}

stats, curves = bh.period(START, END)
show = ['起始日', '結束日', '年數', '總報酬', '年化報酬', '年化波動', '最大回撤', '回撤高點', '回撤低點',
        '回到前高', '夏普', '最好年度', '最差年度', '期末金額', '模擬資料占比']
display(fmt_stats(stats.loc[show]))
plot_equity(curves, f'Buy & Hold {START} ~ {END}'); plt.show()

In [ ]:
#@title 7. 固定長度滾動（每個月一個起點）
YEARS = 10 #@param {type:"number"}

roll = bh.rolling(YEARS)
print(f'{YEARS} 年滾動，每個月初一個起點')
print('\n【年化報酬分布】'); display(fmt_stats(describe(roll, '年化報酬')))
print('\n【最大回撤分布】'); display(fmt_stats(describe(roll, '最大回撤')))
print('\n【最差與最好的區間】'); display(fmt_rows(worst_best(roll)))
for a in ['00631L 台灣50正2', '00675L 加權正2', '大盤（含息）']:
    print(f'{a} 年化報酬勝過 0050 的比例：{win_rate(roll, a, "0050"):.1%}')
plot_windows(roll, title=f'{YEARS:g}-year rolling CAGR'); plt.show()
plot_windows(roll, '最大回撤', title=f'{YEARS:g}-year rolling max drawdown'); plt.show()

In [ ]:
#@title 8. 隨機區間
N = 500 #@param {type:"integer"}
YEARS_MIN = 5 #@param {type:"number"}
YEARS_MAX = 15 #@param {type:"number"}
SEED = 42 #@param {type:"integer"}
#@markdown SEED 固定時每次結果相同；改成 -1 則每次隨機不同。

rnd = bh.random(N, (YEARS_MIN, YEARS_MAX), seed=None if SEED < 0 else SEED)
print(f'隨機 {N} 個區間，長度 {YEARS_MIN}～{YEARS_MAX} 年')
print('\n【年化報酬分布】'); display(fmt_stats(describe(rnd, '年化報酬')))
print('\n【最大回撤分布】'); display(fmt_stats(describe(rnd, '最大回撤')))
print('\n【最差與最好的區間】'); display(fmt_rows(worst_best(rnd)))
plot_windows(rnd, title=f'Random windows ({YEARS_MIN:g}-{YEARS_MAX:g} years) CAGR'); plt.show()